# Context-Aware and Retrieval-Augmented Sarcasm Detection in Social Media

## Notebook 4: Retrieval-augmented evaluation, LLM prompts, and final interpretation

This notebook adds the proposal-facing comparison layer. Retrieval is used as evidence in a calibrated transformer ensemble and also used to build few-shot prompt examples. A pure retrieval-only kNN score is saved as a diagnostic. The LLM section creates reproducible zero-shot and few-shot prompts without inventing scores from a model that was not actually called.


## Configuration

Retrieval uses the shared project constants. `RETRIEVAL_EMBEDDING_MODEL` controls the sentence embedding model. `MAX_RETRIEVAL_TRAIN_ROWS` controls the retrieval index size and `MAX_RETRIEVAL_EVAL_ROWS` controls the evaluation subset size.


In [ ]:
DATASET_SLUG = "/kaggle/input/datasets/danofer/sarcasm"
INPUT_DIR = "/kaggle/input/notebooks/minatahmasebi/"
WORKING_DIR = "/kaggle/working/"
OUTPUT_DIR = "/kaggle/working/"

KAGGLE_NOTEBOOK_INPUT_NAMES = [
    "01-data-and-tfidf-baseline",
    "02-context-free-transformer",
    "03-context-aware-transformer",
    "04-retrieval-and-llm-prompts",
    "01_data_and_tfidf_baseline",
    "02_context_free_transformer",
    "03_context_aware_transformer",
    "04_retrieval_and_llm_prompts",
]

MODEL_NAME = "distilroberta-base"
RANDOM_SEED = 42

DEBUG = False
SAMPLE_SIZE = 20000

MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 3
GRADIENT_ACCUMULATION_STEPS = 1

MAX_TRANSFORMER_TRAIN_ROWS = None
MAX_TRANSFORMER_EVAL_ROWS = None
CPU_MAX_TRANSFORMER_TRAIN_ROWS = 2000
CPU_MAX_TRANSFORMER_EVAL_ROWS = 500
REQUIRE_GPU_FOR_FULL_TRANSFORMER = True

RUN_TFIDF = True
RUN_CONTEXT_FREE_TRANSFORMER = True
RUN_CONTEXT_AWARE_TRANSFORMER = True
RUN_RAG = True
RUN_LLM_PROMPTS = True

RETRIEVAL_EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"
MAX_RETRIEVAL_TRAIN_ROWS = None
MAX_RETRIEVAL_EVAL_ROWS = None
RETRIEVAL_K = 5
ENSEMBLE_CALIBRATION_FRACTION = 0.5
MAX_LLM_PROMPT_ROWS = 100


## Runtime design

The default retrieval evaluation is capped to keep the final comparison reproducible within a normal Kaggle session. `BAAI/bge-base-en-v1.5` is stronger than the earlier MiniLM encoder while still being practical on Kaggle. If runtime is acceptable and you want a heavier retrieval encoder, try `BAAI/bge-large-en-v1.5`; if Kaggle memory or time becomes a problem, return to `BAAI/bge-base-en-v1.5`. The notebook never reports LLM accuracy unless real LLM predictions are supplied.


## Install retrieval dependencies

This simple cell installs the packages Kaggle may not have for retrieval. If FAISS is unavailable, the code falls back to scikit-learn nearest neighbors.


In [ ]:
!pip install -q sentence-transformers faiss-cpu


## Imports and GPU check

Sentence embeddings use a GPU if one is available. The fallback retrieval path works on CPU.


In [ ]:
import json
import os
import random
import shutil
import textwrap
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Retrieval will use CPU.")


## Shared helpers

Files are searched in `/kaggle/working/` and Kaggle notebook-output folders under `/kaggle/input/<notebook-name>/`.


In [ ]:
def kaggle_notebook_input_roots():
    input_root = Path(INPUT_DIR)
    roots = []
    if not input_root.exists():
        return roots

    for notebook_name in KAGGLE_NOTEBOOK_INPUT_NAMES:
        direct_root = input_root / notebook_name
        if direct_root.exists() and direct_root.is_dir():
            roots.append(direct_root)

        for nested_root in input_root.rglob(notebook_name):
            if nested_root.exists() and nested_root.is_dir():
                roots.append(nested_root)

    unique_roots = []
    seen = set()
    for root in roots:
        resolved = root.resolve()
        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(root)
    return unique_roots


def prepare_input_search():
    print("Kaggle input search uses this order:")
    for root in search_roots():
        status = "exists" if root.exists() else "missing"
        print(f"- {root} [{status}]")
    print("Previous notebook outputs are expected at /kaggle/input/<notebook-name>/<output-files>.")


def search_roots():
    roots = [Path(WORKING_DIR)]
    roots.extend(kaggle_notebook_input_roots())
    roots.append(Path(INPUT_DIR))

    unique_roots = []
    seen = set()
    for root in roots:
        resolved = root.resolve() if root.exists() else root
        if resolved not in seen:
            seen.add(resolved)
            unique_roots.append(root)
    return unique_roots


def find_file_recursive(filename):
    matches = []
    roots = search_roots()
    for root in roots:
        if root.exists():
            matches.extend(sorted(root.rglob(filename)))
    if not matches:
        return None

    def rank(path):
        resolved = path.resolve()
        for index, root in enumerate(roots):
            if not root.exists():
                continue
            root_resolved = root.resolve()
            if resolved == root_resolved or root_resolved in resolved.parents:
                return (index, len(str(resolved)))
        return (len(roots), len(str(resolved)))

    matches = sorted(set(matches), key=rank)
    return matches[0]


def require_file(filename, missing_message):
    path = find_file_recursive(filename)
    if path is None:
        print(missing_message)
        print("Searched these locations:")
        for root in search_roots():
            print(f"- {root}")
        raise FileNotFoundError(missing_message)
    print(f"Found {filename}: {path}")
    return path


def copy_to_working(path):
    path = Path(path)
    destination = Path(OUTPUT_DIR) / path.name
    if path.exists() and path.resolve() != destination.resolve():
        shutil.copy2(path, destination)
    return destination


def zip_outputs(zip_name, items):
    zip_path = Path(OUTPUT_DIR) / zip_name
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
        for item in items:
            path = Path(OUTPUT_DIR) / item
            if path.is_file():
                zipf.write(path, arcname=path.name)
            elif path.is_dir():
                for file_path in path.rglob("*"):
                    if file_path.is_file():
                        zipf.write(file_path, arcname=str(file_path.relative_to(Path(OUTPUT_DIR))))
            else:
                print(f"Skipping missing item: {path}")
    print(f"Created ZIP: {zip_path}")
    return zip_path


## Prediction merge helpers

These helpers rebuild `predictions_test.csv` from individual model prediction files when a copied combined file is incomplete.


In [ ]:
def load_prediction_file(filename):
    path = find_file_recursive(filename)
    if path is None:
        return None
    print(f"Loaded predictions from {path}")
    return pd.read_csv(path)


def attach_model_predictions(base_df, predictions_df, pred_col, prob_col):
    base_df[pred_col] = np.nan
    base_df[prob_col] = np.nan
    if predictions_df is None or pred_col not in predictions_df.columns:
        return base_df

    if "example_id" in predictions_df.columns and "example_id" in base_df.columns:
        small = predictions_df[["example_id", pred_col, prob_col]].copy()
        merged = base_df.drop(columns=[pred_col, prob_col]).merge(small, on="example_id", how="left")
        return merged

    if len(predictions_df) == len(base_df):
        base_df[pred_col] = predictions_df[pred_col].values
        if prob_col in predictions_df.columns:
            base_df[prob_col] = predictions_df[prob_col].values
    return base_df


def create_or_update_predictions_test(test_df):
    base = test_df[["example_id", "parent_comment", "comment", "label"]].copy()
    base = base.rename(columns={"label": "true_label"})

    tfidf_predictions = load_prediction_file("tfidf_predictions.csv")
    context_free_predictions = load_prediction_file("context_free_predictions.csv")
    context_aware_predictions = load_prediction_file("context_aware_predictions.csv")

    base = attach_model_predictions(base, tfidf_predictions, "tfidf_pred", "tfidf_prob")
    base = attach_model_predictions(base, context_free_predictions, "context_free_pred", "context_free_prob")
    base = attach_model_predictions(base, context_aware_predictions, "context_aware_pred", "context_aware_prob")

    output_path = Path(OUTPUT_DIR) / "predictions_test.csv"
    base.to_csv(output_path, index=False)
    print(f"Saved {output_path}")
    display(base.head())
    return base


def prediction_correctness(df, pred_col):
    if pred_col not in df.columns:
        return pd.Series(False, index=df.index)
    values = pd.to_numeric(df[pred_col], errors="coerce")
    correct = values.notna() & (values.round().astype("Int64") == df["true_label"])
    return correct.fillna(False)


def create_error_analysis(predictions_df):
    tfidf_available = predictions_df["tfidf_pred"].notna().any() if "tfidf_pred" in predictions_df.columns else False
    context_free_available = predictions_df["context_free_pred"].notna().any() if "context_free_pred" in predictions_df.columns else False
    context_aware_available = predictions_df["context_aware_pred"].notna().any() if "context_aware_pred" in predictions_df.columns else False

    tfidf_correct = prediction_correctness(predictions_df, "tfidf_pred")
    context_free_correct = prediction_correctness(predictions_df, "context_free_pred")
    context_aware_correct = prediction_correctness(predictions_df, "context_aware_pred")

    frames = []

    def add_examples(mask, model_name, pred_col, prob_col, error_type, limit=100):
        if pred_col not in predictions_df.columns:
            return
        subset = predictions_df.loc[mask & predictions_df[pred_col].notna()].head(limit).copy()
        if subset.empty:
            return
        subset["model_name"] = model_name
        subset["predicted_label"] = subset[pred_col]
        subset["probability"] = subset[prob_col] if prob_col in subset.columns else np.nan
        subset["error_type"] = error_type
        frames.append(
            subset[
                [
                    "parent_comment",
                    "comment",
                    "true_label",
                    "predicted_label",
                    "probability",
                    "model_name",
                    "error_type",
                ]
            ]
        )

    if tfidf_available and context_aware_available:
        add_examples(
            (~tfidf_correct) & context_aware_correct,
            "Context-aware Transformer",
            "context_aware_pred",
            "context_aware_prob",
            "TF-IDF wrong but context-aware transformer correct",
        )

    if context_free_available and context_aware_available:
        add_examples(
            (~context_free_correct) & context_aware_correct,
            "Context-aware Transformer",
            "context_aware_pred",
            "context_aware_prob",
            "Context-free transformer wrong but context-aware transformer correct",
        )

    if context_aware_available:
        add_examples(
            ~context_aware_correct,
            "Context-aware Transformer",
            "context_aware_pred",
            "context_aware_prob",
            "Context-aware transformer wrong",
        )

    available_correctness = []
    if tfidf_available:
        available_correctness.append(tfidf_correct)
    if context_free_available:
        available_correctness.append(context_free_correct)
    if context_aware_available:
        available_correctness.append(context_aware_correct)

    if available_correctness:
        all_correct = available_correctness[0].copy()
        all_wrong = ~available_correctness[0].copy()
        for correctness in available_correctness[1:]:
            all_correct = all_correct & correctness
            all_wrong = all_wrong & (~correctness)

        preferred_pred_col = "context_aware_pred" if context_aware_available else ("context_free_pred" if context_free_available else "tfidf_pred")
        preferred_prob_col = preferred_pred_col.replace("_pred", "_prob")
        add_examples(all_wrong, "All available models", preferred_pred_col, preferred_prob_col, "All available models wrong")
        add_examples(all_correct, "All available models", preferred_pred_col, preferred_prob_col, "All available models correct")

    if frames:
        error_analysis = pd.concat(frames, ignore_index=True)
    else:
        error_analysis = pd.DataFrame(
            columns=[
                "parent_comment",
                "comment",
                "true_label",
                "predicted_label",
                "probability",
                "model_name",
                "error_type",
            ]
        )

    output_path = Path(OUTPUT_DIR) / "error_analysis.csv"
    error_analysis.to_csv(output_path, index=False)
    print(f"Saved {output_path}")
    display(error_analysis.head(10))
    return error_analysis


## Load prior outputs

`predictions_test.csv` comes from Notebook 3. If it is missing, run Notebook 3 or attach its output ZIP.


In [ ]:
def find_file_candidates(filename):
    matches = []
    for root in search_roots():
        if root.exists():
            matches.extend(sorted(root.rglob(filename)))
    return sorted(set(matches))


def choose_best_metrics_file(filename="metrics_comparison.csv"):
    candidates = find_file_candidates(filename)
    if not candidates:
        return require_file(filename, "Missing metrics_comparison.csv.")

    scored = []
    for path in candidates:
        try:
            candidate = pd.read_csv(path)
            model_text = " ".join(candidate.get("model_name", pd.Series(dtype=str)).astype(str).str.lower())
            score = len(candidate)
            if "context-aware transformer" in model_text:
                score += 100
            if "context-free transformer" in model_text:
                score += 50
            if "tf-idf" in model_text:
                score += 10
            scored.append((score, path))
        except Exception:
            scored.append((-1, path))

    print("Candidate metrics files:")
    for score, path in sorted(scored, key=lambda item: item[0], reverse=True):
        print(f"- score={score:>3} {path}")

    best_score, best_path = sorted(scored, key=lambda item: item[0], reverse=True)[0]
    if best_score < 100:
        print(
            "Warning: no metrics_comparison.csv candidate contains the context-aware model. "
            "The notebook will try to rebuild the comparison from metric JSON files."
        )
    print(f"Using metrics file: {best_path}")
    return best_path


def choose_best_predictions_file(filename="predictions_test.csv"):
    candidates = find_file_candidates(filename)
    if not candidates:
        return require_file(filename, "Missing predictions_test.csv.")

    scored = []
    for path in candidates:
        try:
            candidate = pd.read_csv(path)
            score = len(candidate.columns)
            if "context_aware_pred" in candidate.columns:
                non_null = candidate["context_aware_pred"].notna().sum()
                score += 100 if non_null > 0 else 0
            if "context_free_pred" in candidate.columns:
                score += 20
            if "tfidf_pred" in candidate.columns:
                score += 10
            scored.append((score, path))
        except Exception:
            scored.append((-1, path))

    print("Candidate prediction files:")
    for score, path in sorted(scored, key=lambda item: item[0], reverse=True):
        print(f"- score={score:>3} {path}")

    best_path = sorted(scored, key=lambda item: item[0], reverse=True)[0][1]
    print(f"Using predictions file: {best_path}")
    return best_path


def rebuild_metrics_from_json():
    metric_files = [
        "tfidf_metrics.json",
        "context_free_metrics.json",
        "context_aware_metrics.json",
    ]
    records = []
    for filename in metric_files:
        path = find_file_recursive(filename)
        if path is None:
            print(f"Missing metric JSON: {filename}")
            continue
        with open(path, "r", encoding="utf-8") as f:
            record = json.load(f)
        records.append(record)
        print(f"Loaded metric JSON: {path}")

    if not records:
        return pd.DataFrame()

    rebuilt = pd.DataFrame(records)
    preferred_columns = [
        "model_name",
        "input_type",
        "accuracy",
        "balanced_accuracy",
        "precision",
        "recall",
        "f1",
        "macro_f1",
        "weighted_f1",
        "roc_auc",
        "average_precision",
        "test_samples",
        "positive_samples",
        "negative_samples",
        "notes",
    ]
    for column in preferred_columns:
        if column not in rebuilt.columns:
            rebuilt[column] = np.nan
    return rebuilt[preferred_columns]


def has_complete_supervised_metrics(metrics_df):
    if "model_name" not in metrics_df.columns:
        return False
    model_text = " ".join(metrics_df["model_name"].astype(str).str.lower())
    return (
        "tf-idf" in model_text
        and "context-free transformer" in model_text
        and "context-aware transformer" in model_text
    )


def has_complete_supervised_predictions(predictions_df):
    required = ["tfidf_pred", "context_free_pred", "context_aware_pred"]
    if not all(column in predictions_df.columns for column in required):
        return False
    return all(predictions_df[column].notna().any() for column in required)


prepare_input_search()

train_path = require_file(
    "train_split.csv",
    "Missing train_split.csv. Please run Notebook 1 first or attach Notebook 1 outputs as a Kaggle dataset.",
)
validation_path = require_file(
    "validation_split.csv",
    "Missing validation_split.csv. Please run Notebook 1 first or attach Notebook 1 outputs as a Kaggle dataset.",
)
test_path = require_file(
    "test_split.csv",
    "Missing test_split.csv. Please run Notebook 1 first or attach Notebook 1 outputs as a Kaggle dataset.",
)
predictions_path = choose_best_predictions_file("predictions_test.csv")
metrics_path = choose_best_metrics_file("metrics_comparison.csv")
error_path = require_file(
    "error_analysis.csv",
    "Missing error_analysis.csv. Please run Notebook 3 first or attach Notebook 3 outputs as a Kaggle dataset.",
)

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
test_df = pd.read_csv(test_path)
predictions_test = pd.read_csv(predictions_path)
metrics_comparison = pd.read_csv(metrics_path)
error_analysis = pd.read_csv(error_path)

if not has_complete_supervised_predictions(predictions_test):
    print(
        "The selected predictions_test.csv is incomplete. "
        "Rebuilding it from tfidf/context-free/context-aware prediction files."
    )
    predictions_test = create_or_update_predictions_test(test_df)

if not has_complete_supervised_metrics(metrics_comparison):
    rebuilt_metrics = rebuild_metrics_from_json()
    if has_complete_supervised_metrics(rebuilt_metrics):
        metrics_comparison = rebuilt_metrics
        metrics_comparison.to_csv(Path(OUTPUT_DIR) / "metrics_comparison.csv", index=False)
        print("Rebuilt metrics_comparison.csv from individual metric JSON files.")
    else:
        print(
            "Warning: supervised metrics are still incomplete. "
            "Attach Notebook 3 outputs that include context_aware_metrics.json."
        )

for source_path in [train_path, validation_path, test_path, error_path]:
    copy_to_working(source_path)

predictions_test.to_csv(Path(OUTPUT_DIR) / "predictions_test.csv", index=False)
metrics_comparison.to_csv(Path(OUTPUT_DIR) / "metrics_comparison.csv", index=False)
print("Saved selected predictions_test.csv and metrics_comparison.csv to the working directory.")

print("Train shape:", train_df.shape)
print("Test predictions shape:", predictions_test.shape)
display(metrics_comparison)
display(predictions_test.head(3))


## Retrieval train and evaluation samples

Retrieval is used as an additional baseline and explanation method. A stratified training sample keeps the vector index practical, and a stratified evaluation sample gives a real score without making this final notebook as expensive as transformer training.


In [ ]:
if len(train_df) > MAX_RETRIEVAL_TRAIN_ROWS:
    retrieval_train_df, _ = train_test_split(
        train_df,
        train_size=MAX_RETRIEVAL_TRAIN_ROWS,
        stratify=train_df["label"],
        random_state=RANDOM_SEED,
    )
    retrieval_train_df = retrieval_train_df.reset_index(drop=True)
    print(f"Using a stratified retrieval sample of {len(retrieval_train_df):,} training rows.")
else:
    retrieval_train_df = train_df.reset_index(drop=True).copy()
    print(f"Using all {len(retrieval_train_df):,} training rows for retrieval.")

if len(test_df) > MAX_RETRIEVAL_EVAL_ROWS:
    retrieval_eval_df, _ = train_test_split(
        test_df,
        train_size=MAX_RETRIEVAL_EVAL_ROWS,
        stratify=test_df["label"],
        random_state=RANDOM_SEED,
    )
    retrieval_eval_df = retrieval_eval_df.reset_index(drop=True)
    print(f"Using a stratified retrieval evaluation sample of {len(retrieval_eval_df):,} test rows.")
else:
    retrieval_eval_df = test_df.reset_index(drop=True).copy()
    print(f"Using all {len(retrieval_eval_df):,} test rows for retrieval evaluation.")


## Build the retrieval index

The preferred path uses the configured `RETRIEVAL_EMBEDDING_MODEL` and FAISS. If either is unavailable, the notebook falls back to scikit-learn nearest neighbors.


In [ ]:
from sklearn.model_selection import train_test_split

def make_retrieval_text(parent_comment, comment):
    return f"{str(parent_comment)} [REPLY] {str(comment)}"


train_retrieval_texts = [
    make_retrieval_text(parent, comment)
    for parent, comment in zip(retrieval_train_df["parent_comment"], retrieval_train_df["comment"])
]

embedding_backend = None
index_backend = None
sentence_model = None
vectorizer = None
train_embeddings = None
faiss_index = None
sklearn_index = None


def build_embeddings(texts):
    global embedding_backend, sentence_model, vectorizer
    try:
        from sentence_transformers import SentenceTransformer

        sentence_model = SentenceTransformer(RETRIEVAL_EMBEDDING_MODEL, device=DEVICE)
        embeddings = sentence_model.encode(
            texts,
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
        )
        embedding_backend = RETRIEVAL_EMBEDDING_MODEL
        return np.asarray(embeddings, dtype="float32")
    except Exception as exc:
        print("Sentence-transformers could not be used:", exc)
        print("Falling back to TF-IDF vectors for retrieval.")
        from sklearn.feature_extraction.text import TfidfVectorizer

        vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
        embedding_backend = "TF-IDF fallback"
        return vectorizer.fit_transform(texts)


def encode_query(texts):
    if embedding_backend == RETRIEVAL_EMBEDDING_MODEL:
        return np.asarray(sentence_model.encode(texts, normalize_embeddings=True), dtype="float32")
    return vectorizer.transform(texts)


if RUN_RAG:
    train_embeddings = build_embeddings(train_retrieval_texts)

    if embedding_backend == RETRIEVAL_EMBEDDING_MODEL:
        try:
            import faiss

            dimension = train_embeddings.shape[1]
            faiss_index = faiss.IndexFlatIP(dimension)
            faiss_index.add(train_embeddings)
            index_backend = "FAISS cosine/IP over normalized sentence embeddings"
        except Exception as exc:
            print("FAISS is unavailable, falling back to sklearn NearestNeighbors:", exc)
            from sklearn.neighbors import NearestNeighbors

            sklearn_index = NearestNeighbors(metric="cosine", algorithm="auto")
            sklearn_index.fit(train_embeddings)
            index_backend = "sklearn NearestNeighbors over sentence embeddings"
    else:
        from sklearn.neighbors import NearestNeighbors

        sklearn_index = NearestNeighbors(metric="cosine", algorithm="auto")
        sklearn_index.fit(train_embeddings)
        index_backend = "sklearn NearestNeighbors over TF-IDF vectors"

    print("Embedding backend:", embedding_backend)
    print("Index backend:", index_backend)
else:
    print("RUN_RAG is False, so retrieval was skipped.")


## Retrieval-augmented classifier

A plain nearest-neighbor vote is a weak diagnostic because semantic similarity does not guarantee the same sarcasm label. The report-facing model therefore uses retrieval as additional evidence on top of the strongest available supervised model. The blend weight and decision threshold are calibrated on one stratified half of the retrieval evaluation sample and reported on the other half.


In [ ]:
def search_retrieval_index(texts, k=RETRIEVAL_K):
    if train_embeddings is None:
        raise RuntimeError("Retrieval index has not been built. Set RUN_RAG=True and run the previous cell.")

    query_embeddings = encode_query(texts)

    if faiss_index is not None:
        scores, indices = faiss_index.search(query_embeddings.astype("float32"), k)
    else:
        distances, indices = sklearn_index.kneighbors(query_embeddings, n_neighbors=k)
        scores = 1 - distances
    return np.asarray(scores), np.asarray(indices)


def predict_from_retrieval(scores, indices):
    neighbor_labels = retrieval_train_df.iloc[indices.reshape(-1)]["label"].to_numpy().reshape(indices.shape)
    weights = np.maximum(scores, 0.0)
    weight_sums = weights.sum(axis=1)
    unweighted_prob = neighbor_labels.mean(axis=1)

    weighted_prob = np.divide(
        (neighbor_labels * weights).sum(axis=1),
        weight_sums,
        out=unweighted_prob.astype(float).copy(),
        where=weight_sums > 0,
    )
    predictions = (weighted_prob >= 0.5).astype(int)
    return predictions, weighted_prob


def retrieve_similar_examples(parent_comment, comment, k=RETRIEVAL_K):
    query_text = make_retrieval_text(parent_comment, comment)
    scores, indices = search_retrieval_index([query_text], k=k)
    scores = scores[0]
    indices = indices[0]

    rows = retrieval_train_df.iloc[indices][["parent_comment", "comment", "label"]].copy()
    rows["similarity_score"] = scores
    return rows.reset_index(drop=True)


if RUN_RAG:
    retrieval_eval_texts = [
        make_retrieval_text(parent, comment)
        for parent, comment in zip(retrieval_eval_df["parent_comment"], retrieval_eval_df["comment"])
    ]
    retrieval_scores, retrieval_indices = search_retrieval_index(retrieval_eval_texts, k=RETRIEVAL_K)
    retrieval_pred, retrieval_prob = predict_from_retrieval(retrieval_scores, retrieval_indices)

    y_true = retrieval_eval_df["label"].astype(int).to_numpy()

    def make_metric_row(model_name, input_type, labels, predictions, scores, notes):
        return {
            "model_name": model_name,
            "input_type": input_type,
            "accuracy": float(accuracy_score(labels, predictions)),
            "balanced_accuracy": float(balanced_accuracy_score(labels, predictions)),
            "precision": float(precision_score(labels, predictions, zero_division=0)),
            "recall": float(recall_score(labels, predictions, zero_division=0)),
            "f1": float(f1_score(labels, predictions, zero_division=0)),
            "macro_f1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
            "weighted_f1": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
            "roc_auc": float(roc_auc_score(labels, scores)),
            "average_precision": float(average_precision_score(labels, scores)),
            "test_samples": int(len(labels)),
            "positive_samples": int((labels == 1).sum()),
            "negative_samples": int((labels == 0).sum()),
            "notes": notes,
        }

    retrieval_knn_metrics = make_metric_row(
        model_name=f"Retrieval-only kNN diagnostic (k={RETRIEVAL_K})",
        input_type="parent_comment + comment + retrieved examples",
        labels=y_true,
        predictions=retrieval_pred,
        scores=retrieval_prob,
        notes=(
            f"Diagnostic retrieval-only vote using {embedding_backend} with {index_backend}. "
            f"Predictions use similarity-weighted vote over k={RETRIEVAL_K} nearest training examples. "
            "This is not expected to beat fine-tuned transformers because semantic similarity does not always imply the same sarcasm label."
        ),
    )

    retrieval_augmented_predictions = retrieval_eval_df[
        ["example_id", "parent_comment", "comment", "label"]
    ].rename(columns={"label": "true_label"}).copy()
    retrieval_augmented_predictions["retrieval_knn_pred"] = retrieval_pred
    retrieval_augmented_predictions["retrieval_knn_prob"] = retrieval_prob

    prediction_columns = [
        "example_id",
        "tfidf_prob",
        "context_free_prob",
        "context_aware_prob",
    ]
    available_prediction_columns = [
        column for column in prediction_columns if column in predictions_test.columns
    ]
    supervised_probs = predictions_test[available_prediction_columns].copy()
    retrieval_augmented_predictions = retrieval_augmented_predictions.merge(
        supervised_probs,
        on="example_id",
        how="left",
    )

    base_prob_col = None
    for candidate_col in ["context_aware_prob", "context_free_prob", "tfidf_prob"]:
        if (
            candidate_col in retrieval_augmented_predictions.columns
            and retrieval_augmented_predictions[candidate_col].notna().any()
        ):
            base_prob_col = candidate_col
            break

    base_report_metrics = None
    if base_prob_col is not None and len(np.unique(y_true)) == 2:
        from sklearn.linear_model import LogisticRegression
        from sklearn.pipeline import make_pipeline
        from sklearn.preprocessing import StandardScaler

        row_indices = np.arange(len(retrieval_augmented_predictions))
        calibration_indices, report_indices = train_test_split(
            row_indices,
            train_size=ENSEMBLE_CALIBRATION_FRACTION,
            stratify=y_true,
            random_state=RANDOM_SEED,
        )
        base_prob = retrieval_augmented_predictions[base_prob_col].astype(float).to_numpy()

        neighbor_labels = retrieval_train_df.iloc[retrieval_indices.reshape(-1)]["label"].to_numpy().reshape(retrieval_indices.shape)
        top1_label = neighbor_labels[:, 0]
        top1_score = retrieval_scores[:, 0]
        score_mean = retrieval_scores.mean(axis=1)
        score_std = retrieval_scores.std(axis=1)
        score_gap = retrieval_scores[:, 0] - retrieval_scores[:, 1]
        positive_neighbor_count = neighbor_labels.sum(axis=1)

        feature_columns = []
        feature_arrays = []
        for prob_col in ["tfidf_prob", "context_free_prob", "context_aware_prob"]:
            if (
                prob_col in retrieval_augmented_predictions.columns
                and retrieval_augmented_predictions[prob_col].notna().any()
            ):
                feature_columns.append(prob_col)
                feature_arrays.append(retrieval_augmented_predictions[prob_col].astype(float).to_numpy())

        retrieval_feature_map = {
            "retrieval_knn_prob": retrieval_prob,
            "retrieval_top1_label": top1_label,
            "retrieval_top1_score": top1_score,
            "retrieval_score_mean": score_mean,
            "retrieval_score_std": score_std,
            "retrieval_score_gap": score_gap,
            "retrieval_positive_neighbor_count": positive_neighbor_count,
        }
        for feature_name, values in retrieval_feature_map.items():
            feature_columns.append(feature_name)
            feature_arrays.append(values.astype(float))
            retrieval_augmented_predictions[feature_name] = values

        stacked_features = np.vstack(feature_arrays).T
        stacked_features = np.nan_to_num(stacked_features, nan=0.5, posinf=1.0, neginf=0.0)
        stacker = make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_SEED),
        )
        stacker.fit(stacked_features[calibration_indices], y_true[calibration_indices])
        stacker_prob = stacker.predict_proba(stacked_features)[:, 1]

        def best_threshold_for_accuracy(scores, labels):
            best_accuracy = -1.0
            best_threshold = 0.5
            for threshold in np.linspace(0.25, 0.75, 101):
                predictions = (scores >= threshold).astype(int)
                value = accuracy_score(labels, predictions)
                if value > best_accuracy:
                    best_accuracy = value
                    best_threshold = float(threshold)
            return best_threshold

        base_threshold = best_threshold_for_accuracy(
            base_prob[calibration_indices],
            y_true[calibration_indices],
        )
        stacker_threshold = best_threshold_for_accuracy(
            stacker_prob[calibration_indices],
            y_true[calibration_indices],
        )

        base_pred = (base_prob >= base_threshold).astype(int)
        stacker_pred = (stacker_prob >= stacker_threshold).astype(int)

        base_report_metrics = make_metric_row(
            model_name=f"Base supervised model on retrieval report split ({base_prob_col})",
            input_type="supervised model probability only",
            labels=y_true[report_indices],
            predictions=base_pred[report_indices],
            scores=base_prob[report_indices],
            notes=(
                f"Same disjoint report split as retrieval stacker. Threshold={base_threshold:.2f} "
                "was selected on the calibration split for accuracy."
            ),
        )

        stacker_report_metrics = make_metric_row(
            model_name="Retrieval-augmented calibrated stacker",
            input_type="supervised probabilities + retrieval-neighbor features",
            labels=y_true[report_indices],
            predictions=stacker_pred[report_indices],
            scores=stacker_prob[report_indices],
            notes=(
                f"Calibrated logistic stacker using features: {', '.join(feature_columns)}. "
                f"Threshold={stacker_threshold:.2f} was selected on the calibration split for accuracy, "
                "then evaluated on the disjoint report split."
            ),
        )

        final_prob = stacker_prob
        final_pred = stacker_pred
        retrieval_augmented_predictions["retrieval_augmented_prob"] = final_prob
        retrieval_augmented_predictions["retrieval_augmented_pred"] = final_pred
        retrieval_augmented_predictions["base_report_prob"] = base_prob
        retrieval_augmented_predictions["base_report_pred"] = base_pred
        retrieval_augmented_predictions["evaluation_split"] = "report"
        retrieval_augmented_predictions.loc[calibration_indices, "evaluation_split"] = "calibration"

        retrieval_metrics = stacker_report_metrics
        if stacker_report_metrics["accuracy"] < base_report_metrics["accuracy"]:
            print(
                "Warning: retrieval features did not improve over the base supervised model on the report split. "
                "Report this as a negative retrieval result rather than claiming an improvement."
            )
    else:
        retrieval_augmented_predictions["retrieval_augmented_prob"] = retrieval_prob
        retrieval_augmented_predictions["retrieval_augmented_pred"] = retrieval_pred
        retrieval_augmented_predictions["evaluation_split"] = "report"
        retrieval_metrics = retrieval_knn_metrics
        retrieval_metrics["model_name"] = f"Retrieval-augmented kNN (k={RETRIEVAL_K})"
        retrieval_metrics["notes"] = (
            retrieval_metrics["notes"]
            + " No supervised probability column was available, so the notebook could not build the transformer-retrieval ensemble."
        )

    retrieval_augmented_predictions.to_csv(
        Path(OUTPUT_DIR) / "retrieval_augmented_predictions.csv",
        index=False,
    )

    with open(Path(OUTPUT_DIR) / "retrieval_metrics.json", "w", encoding="utf-8") as f:
        json.dump(retrieval_metrics, f, indent=2)
    with open(Path(OUTPUT_DIR) / "retrieval_knn_diagnostic_metrics.json", "w", encoding="utf-8") as f:
        json.dump(retrieval_knn_metrics, f, indent=2)

    diagnostic_rows = [retrieval_knn_metrics]
    if base_report_metrics is not None:
        diagnostic_rows.append(base_report_metrics)
    diagnostic_rows.append(retrieval_metrics)
    retrieval_metrics_df = pd.DataFrame(diagnostic_rows)
    display(retrieval_metrics_df)
    print("Saved retrieval_augmented_predictions.csv, retrieval_metrics.json, and retrieval_knn_diagnostic_metrics.json")
else:
    retrieval_augmented_predictions = pd.DataFrame()
    retrieval_metrics = None
    print("RUN_RAG is False, so retrieval evaluation was skipped.")


retrieval_frames = []
if RUN_RAG:
    demo_examples = predictions_test.head(5).copy()
    for query_number, (_, row) in enumerate(demo_examples.iterrows(), start=1):
        retrieved = retrieve_similar_examples(row["parent_comment"], row["comment"], k=5)
        retrieved.insert(0, "query_number", query_number)
        retrieved.insert(1, "query_parent_comment", row["parent_comment"])
        retrieved.insert(2, "query_comment", row["comment"])
        retrieved.insert(3, "query_true_label", row["true_label"])
        retrieved.insert(4, "query_context_aware_pred", row.get("context_aware_pred", np.nan))
        retrieval_frames.append(retrieved)

        print("\nQuery", query_number)
        print("Parent:", str(row["parent_comment"])[:300])
        print("Reply:", str(row["comment"])[:300])
        display(retrieved[["parent_comment", "comment", "label", "similarity_score"]])

    retrieval_examples = pd.concat(retrieval_frames, ignore_index=True)
else:
    retrieval_examples = pd.DataFrame(
        columns=[
            "query_number",
            "query_parent_comment",
            "query_comment",
            "query_true_label",
            "query_context_aware_pred",
            "parent_comment",
            "comment",
            "label",
            "similarity_score",
        ]
    )

retrieval_examples.to_csv(Path(OUTPUT_DIR) / "retrieval_examples.csv", index=False)
print("Saved retrieval_examples.csv")


## LLM prompt builders

No paid API is called. These functions only write prompts for optional zero-shot and few-shot comparison.


In [ ]:
def build_zero_shot_prompt(parent_comment, comment):
    return textwrap.dedent(
        f'''
        You are judging sarcasm in a Reddit conversation.

        Parent comment:
        {parent_comment}

        Target reply:
        {comment}

        Is the target reply sarcastic? Answer 0 or 1 and explain briefly.
        Use 1 for sarcastic and 0 for non-sarcastic.
        '''
    ).strip()


def build_few_shot_prompt(parent_comment, comment, retrieved_examples):
    example_blocks = []
    for i, (_, example) in enumerate(retrieved_examples.head(3).iterrows(), start=1):
        example_blocks.append(
            textwrap.dedent(
                f'''
                Example {i}
                Parent comment: {example["parent_comment"]}
                Target reply: {example["comment"]}
                Label: {int(example["label"])}
                '''
            ).strip()
        )

    examples_text = "\n\n".join(example_blocks) if example_blocks else "No retrieved examples are available."
    return textwrap.dedent(
        f'''
        You are judging sarcasm in a Reddit conversation.

        Here are similar labeled examples:

        {examples_text}

        Now classify this new case.

        Parent comment:
        {parent_comment}

        Target reply:
        {comment}

        Is the target reply sarcastic? Answer 0 or 1 and explain briefly.
        Use 1 for sarcastic and 0 for non-sarcastic.
        '''
    ).strip()


## Save prompt examples

The text file is easy to read, and the CSV is easier to analyze later.


In [ ]:
prompt_rows = []
prompt_text_blocks = []

if RUN_LLM_PROMPTS:
    if len(predictions_test) > MAX_LLM_PROMPT_ROWS:
        prompt_examples, _ = train_test_split(
            predictions_test,
            train_size=MAX_LLM_PROMPT_ROWS,
            stratify=predictions_test["true_label"],
            random_state=RANDOM_SEED,
        )
        prompt_examples = prompt_examples.reset_index(drop=True)
    else:
        prompt_examples = predictions_test.reset_index(drop=True).copy()

    for query_number, (_, row) in enumerate(prompt_examples.iterrows(), start=1):
        if RUN_RAG and train_embeddings is not None:
            retrieved = retrieve_similar_examples(row["parent_comment"], row["comment"], k=5)
        else:
            retrieved = pd.DataFrame(columns=["parent_comment", "comment", "label", "similarity_score"])

        zero_prompt = build_zero_shot_prompt(row["parent_comment"], row["comment"])
        few_prompt = build_few_shot_prompt(row["parent_comment"], row["comment"], retrieved)

        prompt_rows.append(
            {
                "query_number": query_number,
                "parent_comment": row["parent_comment"],
                "comment": row["comment"],
                "true_label": row["true_label"],
                "zero_shot_prompt": zero_prompt,
                "few_shot_prompt": few_prompt,
            }
        )

        if query_number <= 5:
            prompt_text_blocks.append(
                f"==================== Query {query_number}: zero-shot ====================\n"
                + zero_prompt
                + "\n\n"
                + f"==================== Query {query_number}: few-shot ====================\n"
                + few_prompt
            )

    llm_prompt_examples = pd.DataFrame(prompt_rows)
    llm_prediction_template = llm_prompt_examples[
        ["query_number", "parent_comment", "comment", "true_label"]
    ].copy()
    llm_prediction_template["zero_shot_pred"] = pd.NA
    llm_prediction_template["zero_shot_prob"] = pd.NA
    llm_prediction_template["few_shot_pred"] = pd.NA
    llm_prediction_template["few_shot_prob"] = pd.NA
else:
    llm_prompt_examples = pd.DataFrame(
        columns=[
            "query_number",
            "parent_comment",
            "comment",
            "true_label",
            "zero_shot_prompt",
            "few_shot_prompt",
        ]
    )
    llm_prediction_template = pd.DataFrame(
        columns=[
            "query_number",
            "parent_comment",
            "comment",
            "true_label",
            "zero_shot_pred",
            "zero_shot_prob",
            "few_shot_pred",
            "few_shot_prob",
        ]
    )

llm_prompt_examples.to_csv(Path(OUTPUT_DIR) / "llm_prompt_examples.csv", index=False)
llm_prompt_examples.to_csv(Path(OUTPUT_DIR) / "llm_prompt_eval.csv", index=False)
llm_prediction_template.to_csv(Path(OUTPUT_DIR) / "llm_predictions_template.csv", index=False)
with open(Path(OUTPUT_DIR) / "llm_prompt_examples.txt", "w", encoding="utf-8") as f:
    f.write("\n\n".join(prompt_text_blocks))

print("Saved llm_prompt_examples.csv, llm_prompt_eval.csv, llm_predictions_template.csv, and llm_prompt_examples.txt")
display(llm_prompt_examples.head(2))


## Update comparison table

The proposal includes retrieval-augmented and LLM-based comparison. Retrieval is evaluated here with real predictions. LLM metrics are added only when a real `llm_predictions.csv` file is available; otherwise the notebook keeps the LLM section qualitative and reproducible.


In [ ]:
def evaluate_binary_predictions(model_name, input_type, y_true, y_pred, y_score=None, notes=""):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    row = {
        "model_name": model_name,
        "input_type": input_type,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "roc_auc": np.nan,
        "average_precision": np.nan,
        "test_samples": int(len(y_true)),
        "positive_samples": int((y_true == 1).sum()),
        "negative_samples": int((y_true == 0).sum()),
        "notes": notes,
    }
    if y_score is not None:
        y_score = np.asarray(y_score).astype(float)
        if len(np.unique(y_true)) == 2:
            row["roc_auc"] = float(roc_auc_score(y_true, y_score))
            row["average_precision"] = float(average_precision_score(y_true, y_score))
    return row


comparison_rows = metrics_comparison.copy()

if retrieval_metrics is not None:
    comparison_rows = comparison_rows[
        comparison_rows["model_name"] != retrieval_metrics["model_name"]
    ].copy()
    comparison_rows = pd.concat(
        [comparison_rows, pd.DataFrame([retrieval_metrics])],
        ignore_index=True,
    )

llm_metrics = []
llm_predictions_path = find_file_recursive("llm_predictions.csv")
if llm_predictions_path is not None:
    llm_predictions = pd.read_csv(llm_predictions_path)
    if "true_label" not in llm_predictions.columns:
        raise ValueError("llm_predictions.csv must include a true_label column.")

    llm_specs = [
        ("LLM zero-shot classifier", "parent_comment + comment prompt", "zero_shot_pred", "zero_shot_prob"),
        ("LLM few-shot retrieval-augmented classifier", "parent_comment + comment + retrieved examples prompt", "few_shot_pred", "few_shot_prob"),
    ]

    for model_name, input_type, pred_col, prob_col in llm_specs:
        if pred_col in llm_predictions.columns:
            y_score = llm_predictions[prob_col] if prob_col in llm_predictions.columns else None
            row = evaluate_binary_predictions(
                model_name=model_name,
                input_type=input_type,
                y_true=llm_predictions["true_label"],
                y_pred=llm_predictions[pred_col],
                y_score=y_score,
                notes=(
                    f"Evaluated from user-supplied {llm_predictions_path.name}. "
                    "The notebook does not call a paid LLM API."
                ),
            )
            llm_metrics.append(row)

    if llm_metrics:
        comparison_rows = comparison_rows[
            ~comparison_rows["model_name"].isin([row["model_name"] for row in llm_metrics])
        ].copy()
        comparison_rows = pd.concat(
            [comparison_rows, pd.DataFrame(llm_metrics)],
            ignore_index=True,
        )
        pd.DataFrame(llm_metrics).to_csv(Path(OUTPUT_DIR) / "llm_metrics.csv", index=False)
        print("Saved llm_metrics.csv")
    else:
        print("llm_predictions.csv was found, but no supported prediction columns were present.")
else:
    print("No llm_predictions.csv found. LLM prompts are saved, but no LLM accuracy is reported.")

metric_columns = list(metrics_comparison.columns)
for column in metric_columns:
    if column not in comparison_rows.columns:
        comparison_rows[column] = np.nan
extra_columns = [column for column in comparison_rows.columns if column not in metric_columns]
metrics_comparison = comparison_rows[metric_columns + extra_columns].copy()
metrics_comparison.to_csv(Path(OUTPUT_DIR) / "metrics_comparison.csv", index=False)

print("Saved updated metrics_comparison.csv")
display(metrics_comparison)


## Final conclusion

This conclusion is generated from `metrics_comparison.csv`, and it does not hide a weak result.


In [ ]:
def find_model_row(metrics_df, phrase):
    mask = metrics_df["model_name"].astype(str).str.lower().str.contains(phrase.lower(), regex=False)
    if mask.any():
        return metrics_df.loc[mask].iloc[0]
    return None


def print_best(metrics_df, metric, label):
    if metric not in metrics_df.columns or metrics_df[metric].dropna().empty:
        print(f"Best {label}: not available")
        return None
    row = metrics_df.dropna(subset=[metric]).sort_values(metric, ascending=False).iloc[0]
    print(f"Best {label}: {row['model_name']} with {metric} = {row[metric]:.4f}")
    return row


usable_metrics = metrics_comparison.copy()
print("Final project conclusion")
print("========================")
print_best(usable_metrics, "accuracy", "accuracy")
print_best(usable_metrics, "balanced_accuracy", "balanced accuracy")
print_best(usable_metrics, "precision", "precision")
print_best(usable_metrics, "recall", "recall")
print_best(usable_metrics, "f1", "F1")
print_best(usable_metrics, "macro_f1", "macro-F1")
print_best(usable_metrics, "weighted_f1", "weighted-F1")
print_best(usable_metrics, "roc_auc", "ROC-AUC")
print_best(usable_metrics, "average_precision", "average precision")

context_free = find_model_row(usable_metrics, "context-free transformer")
context_aware = find_model_row(usable_metrics, "context-aware transformer")

if context_free is not None and context_aware is not None:
    f1_difference = context_aware["f1"] - context_free["f1"]
    macro_f1_difference = context_aware["macro_f1"] - context_free["macro_f1"]
    print(f"Context-aware minus context-free F1: {f1_difference:+.4f}")
    print(f"Context-aware minus context-free macro-F1: {macro_f1_difference:+.4f}")

    if f1_difference > 0:
        print("The context-aware transformer beats the context-free transformer by F1 in this run.")
    else:
        print("The context-aware transformer does not beat the context-free transformer by F1 in this run.")
        print(
            "A fair explanation is that context may be noisy, target comments may already contain sarcasm markers, "
            "max_length may truncate useful context, or the training/sample size may be limited."
        )
else:
    print("The context-free and context-aware transformer rows are not both available, so their direct comparison cannot be computed.")

if Path(OUTPUT_DIR, "retrieval_examples.csv").exists():
    print("Retrieval is evaluated as an additional baseline and also used for explanation examples.")


## Save final outputs

This ZIP contains the retrieval examples, prompt examples, and final comparison files.


In [ ]:
final_outputs = [
    "retrieval_augmented_predictions.csv",
    "retrieval_metrics.json",
    "retrieval_knn_diagnostic_metrics.json",
    "retrieval_examples.csv",
    "llm_metrics.csv",
    "llm_prompt_examples.csv",
    "llm_prompt_eval.csv",
    "llm_predictions_template.csv",
    "llm_prompt_examples.txt",
    "predictions_test.csv",
    "metrics_comparison.csv",
    "error_analysis.csv",
]

zip_outputs("sarcasm_final_outputs.zip", final_outputs)
print("Notebook 4 complete.")
